# Data Ingestion & Vector Stores — RouteMind

**Scenario:** Alongside the shipments/drivers database, Portside Logistics has a handful of policy documents that RouteMind needs to search semantically instead of with exact SQL lookups: the driver handbook, the safety and compliance guide, the customer SLA policy, hazmat handling procedures, and the vehicle maintenance manual. This notebook covers the basic building blocks for a simple local retrieval pipeline over documents like these:

- Document ingestion from PDFs, web pages, and CSVs
- Metadata tagging
- Chunking strategies
- Local Hugging Face embeddings on CPU (with an offline-safe fallback)
- Chroma local persistence
- FAISS in-memory vector search
- Optional retriever tracing in LangSmith

LangChain's retrieval docs describe the pipeline as modular, with loaders, splitters, embeddings, and vector stores that can be swapped without rewriting the app. The embeddings docs also note that local CPU models can be used for short queries, and the Chroma docs show both in-memory and persistent local usage.

## Learning goals

By the end of this notebook, you should be able to:

1. Load documents from a few common source types.
2. Attach useful metadata to each document.
3. Split text into chunks.
4. Build embeddings locally on CPU (or fall back gracefully without network access).
5. Store embeddings in Chroma with local persistence.
6. Create a temporary in-memory FAISS index.
7. Trace retrieval steps later with LangSmith if needed.

## 1) Install packages

Run this once in a notebook cell:

```bash
pip install -U pypdf pandas beautifulsoup4 requests sentence-transformers langchain langchain-community langchain-text-splitters langchain-huggingface langchain-chroma chromadb faiss-cpu
```

If you later want LangSmith retriever traces, add:

```bash
pip install -U langsmith
```

In [ ]:
%pip install -qU pypdf pandas beautifulsoup4 requests sentence-transformers langchain langchain-community langchain-text-splitters langchain-huggingface langchain-chroma chromadb faiss-cpu langsmith

## 2) What counts as a document?

In LangChain retrieval, document loaders ingest data from external sources and return standardized `Document` objects. That keeps the rest of the pipeline consistent whether the source is a PDF policy document, a web page, or a CSV export of shipment data.

In [ ]:
from typing import List
from pathlib import Path
from datetime import datetime

import requests
import pandas as pd
from bs4 import BeautifulSoup

from langchain_core.documents import Document

## 3) Simple ingestion helpers

These helpers keep the notebook basic:

- PDF → text extraction, for policy documents like the driver handbook
- Web page → readable text, for anything published on Portside's internal knowledge base
- CSV → rows as small documents, for exports like a shipments or support-ticket dump

Each document gets metadata so you can filter and inspect sources later.

In [ ]:
def load_pdf(path: str) -> List[Document]:
    from pypdf import PdfReader

    reader = PdfReader(path)
    docs: List[Document] = []
    for page_num, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        if text.strip():
            docs.append(
                Document(
                    page_content=text,
                    metadata={
                        "source": path,
                        "type": "pdf",
                        "page": page_num + 1,
                    },
                )
            )
    return docs


def load_web_page(url: str) -> List[Document]:
    response = requests.get(url, timeout=20)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    text = " ".join(soup.get_text(" ").split())
    return [
        Document(
            page_content=text,
            metadata={"source": url, "type": "web"},
        )
    ]


def load_csv(path: str) -> List[Document]:
    df = pd.read_csv(path)
    docs: List[Document] = []
    for idx, row in df.iterrows():
        row_text = " | ".join(f"{col}: {row[col]}" for col in df.columns)
        docs.append(
            Document(
                page_content=row_text,
                metadata={
                    "source": path,
                    "type": "csv",
                    "row_index": int(idx),
                },
            )
        )
    return docs

## 4) Metadata tagging

Metadata makes retrieval better because it lets you inspect and filter documents by source, page number, row index, or file type.

A simple metadata pattern looks like this:

- `source`
- `type`
- `page`
- `row_index`
- `created_at`
- `topic`

In [ ]:
sample_docs = [
    Document(
        page_content="Drivers must take a 30 minute break after 5 hours of continuous driving.",
        metadata={"source": "driver_handbook.pdf", "type": "policy", "topic": "hours_of_service"},
    ),
    Document(
        page_content="Standard service targets 3 to 5 business days between the origin depot and the destination city.",
        metadata={"source": "customer_sla_policy.pdf", "type": "policy", "topic": "delivery_windows"},
    ),
]

sample_docs

## 5) Chunking strategies

LangChain's retrieval docs say text splitters break large documents into smaller chunks that can be retrieved individually and fit into a model's context window.

For basics, keep this simple:

- Character-based chunking: easy and predictable
- Token-based chunking: better when you care about model context size
- Semantic chunking: useful later when you want smarter boundary selection

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
)

chunks = text_splitter.split_documents(sample_docs)

print('Chunks:', len(chunks))
for i, chunk in enumerate(chunks, 1):
    print(f'--- Chunk {i} ---')
    print(chunk.page_content)
    print(chunk.metadata)

## 6) Hugging Face embeddings on CPU

LangChain's Hugging Face docs say `HuggingFaceEmbeddings` can run open source embedding models locally. The embeddings docs also note that local CPU models can be used for short queries, while hosted APIs add network latency.

This cell tries the real model first. If it can't be downloaded -- no internet access to Hugging Face, or the package isn't installed -- it falls back to a small deterministic hashing-based embedding, so the rest of the notebook still runs end to end. Swap back to the real model on a machine with internet access for genuinely semantic results.

In [ ]:
import hashlib
from langchain_core.embeddings import Embeddings


class OfflineHashEmbeddings(Embeddings):
    """Deterministic, dependency-free embeddings for sandboxes without network
    access to Hugging Face. Uses a fixed-dimension hashing trick (a simplified
    bag-of-words / feature-hashing model) so Chroma/FAISS can still perform
    real similarity search based on shared vocabulary. Retrieval quality is
    noticeably rougher than a real sentence-transformers model -- this exists
    purely so the pipeline still runs offline, not as a production embedding."""

    def __init__(self, dim: int = 384):
        self.dim = dim

    def _embed(self, text: str):
        vec = [0.0] * self.dim
        for token in text.lower().split():
            h = int(hashlib.sha256(token.encode("utf-8")).hexdigest(), 16)
            vec[h % self.dim] += 1.0
        norm = sum(v * v for v in vec) ** 0.5
        if norm > 0:
            vec = [v / norm for v in vec]
        return vec

    def embed_documents(self, texts):
        return [self._embed(t) for t in texts]

    def embed_query(self, text):
        return self._embed(text)


try:
    from langchain_huggingface import HuggingFaceEmbeddings
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    embeddings.embed_query("connectivity check")  # forces the lazy model load now, not later
    print("Embeddings ready: HuggingFace all-MiniLM-L6-v2 (live)")
except Exception as e:
    embeddings = OfflineHashEmbeddings(dim=384)
    print(f"Embeddings ready: offline hashing fallback ({type(e).__name__}: sentence-transformers "
          f"could not be loaded in this environment)")

## 7) Chroma local persistence

Chroma can run locally without a server. The LangChain docs show `persist_directory` for saving data across runs, and the underlying Chroma client also supports `PersistentClient(path=...)` for local persistence.

In [ ]:
from langchain_chroma import Chroma

chroma_dir = './chroma_data'

chroma_store = Chroma(
    collection_name='routemind_policy_docs',
    embedding_function=embeddings,
    persist_directory=chroma_dir,
)

chroma_store.add_documents(chunks)

print('Saved chunks to Chroma at', chroma_dir)

In [ ]:
results = chroma_store.similarity_search('What are the rules for driver rest breaks?', k=2)

for i, doc in enumerate(results, 1):
    print(f'--- Result {i} ---')
    print(doc.page_content)
    print(doc.metadata)

## 8) FAISS in-memory vector search

FAISS is a good choice when you want a temporary, fast local index during a notebook session or test run.

Here we build it in memory from the same chunks.

In [ ]:
from langchain_community.vectorstores import FAISS

faiss_index = FAISS.from_documents(chunks, embeddings)

faiss_results = faiss_index.similarity_search('How long does standard delivery take?', k=2)

for i, doc in enumerate(faiss_results, 1):
    print(f'--- FAISS Result {i} ---')
    print(doc.page_content)
    print(doc.metadata)

## 9) In-memory vs persistent vector stores

### Use Chroma persistence when:
- you want the index saved across notebook runs
- you are building a local app that needs durable search -- like RouteMind's policy lookup, which should survive a restart
- you want a simple directory-based store

### Use FAISS in memory when:
- you only need the index for the current process
- you want very fast temporary semantic search
- you are experimenting or testing

## 10) Document ingestion examples you can adapt

The workflow below is the basic pattern:

1. Load content.
2. Attach metadata.
3. Split text.
4. Embed the chunks.
5. Store them in a vector database.
6. Retrieve relevant chunks later.

In [ ]:
# Example layout for Portside Logistics' local files
# handbook_docs = load_pdf('docs/driver_handbook.pdf')
# sla_docs = load_pdf('docs/customer_sla_policy.pdf')
# web_docs = load_web_page('https://internal.portside-logistics.example/policies/hazmat')
# ticket_export_docs = load_csv('data/support_tickets_export.csv')
# all_docs = handbook_docs + sla_docs + web_docs + ticket_export_docs
# chunks = text_splitter.split_documents(all_docs)
print('Loader templates are ready.')

## 11) Optional retriever tracing

LangSmith has dedicated retriever trace rendering so you can inspect retrieved documents and diagnose retrieval issues. The retriever trace page says these traces give document-level visibility into a RAG pipeline.

For later use, keep tracing enabled with:

```env
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=...
LANGSMITH_PROJECT=routemind-retrieval
```

In [ ]:
# Skeleton for later LangSmith tracing
# from langsmith import trace
#
# with trace('routemind-retrieval-demo', run_type='retriever', project_name='routemind-retrieval'):
#     docs = chroma_store.similarity_search('driver rest break policy', k=3)
#     print(docs)

print('Tracing skeleton ready for later.')

## 12) Keep chunking simple at the start

For a first pass, use character-based splitting with modest overlap. That gives you a predictable baseline before you experiment with token-aware or semantic splitting later.

That is enough for basic indexing, retrieval testing, and notebook demos -- the guided lab that follows this notebook compares three chunking configurations against Portside's real policy PDFs.

## Key takeaways

- Load documents into a common `Document` format.
- Attach metadata as early as possible.
- Start with simple chunking before trying advanced split logic.
- Use `HuggingFaceEmbeddings` on CPU for local embeddings, with a deterministic offline fallback when there's no network access.
- Use Chroma persistence when you want a saved local index.
- Use FAISS in memory for fast temporary search.
- Add LangSmith retriever tracing later when you want visibility into retrieval steps.

## Verification status in this environment

| Component | Status |
|---|---|
| Document/metadata basics, character chunking | Executed and verified |
| Chroma persistence + FAISS in-memory retrieval | Executed and verified, using the offline hashing embeddings fallback |
| `HuggingFaceEmbeddings` (`all-MiniLM-L6-v2`) | Not verified here -- this sandbox has no network route to `huggingface.co`. Falls back automatically; swap back in on a machine with internet access. |
| LangSmith retriever tracing (section 11) | Left as a skeleton, matching the original design -- not executed either way |

## References

- Document loaders: https://docs.langchain.com/oss/python/integrations/document_loaders/index
- Retrieval building blocks: https://docs.langchain.com/oss/python/langchain/retrieval#building-blocks
- Vector stores: https://docs.langchain.com/oss/python/integrations/vectorstores/index
- Hugging Face embeddings: https://docs.langchain.com/oss/python/integrations/providers/huggingface#huggingfaceembeddings
- Embeddings latency: https://docs.langchain.com/oss/python/integrations/embeddings/index#latency
- LangSmith retriever traces: https://docs.langchain.com/langsmith/log-retriever-trace